<a href="https://colab.research.google.com/github/willychang21/mini-sglang/blob/main/notebooks/gpu_benchmarks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mini-sglang on a GPU: attention backends, CUDA Graphs, benchmarks

Runs on Colab (T4 or A100) or any CUDA machine. It

1. installs this repo (branch `main`) and downloads `Qwen/Qwen3-0.6B`,
2. runs the **whole test suite on the GPU** (incl. `tests/test_gpu.py`, which is skipped on CPU: real Triton kernels, CUDA Graph capture/replay, fp32 greedy == HuggingFace),
3. runs **every script in `benchmarks/`** with the `torch` and `triton` attention backends, with and without CUDA Graphs,
4. records a **`torch.profiler` trace of one decode step** for each configuration.

Nothing here has been run yet: the output cells are intentionally empty. Each section has a *What to expect* note so you can tell whether the numbers make sense.

> **T4 note**: the T4 (sm_75) has no bfloat16 tensor cores, so the notebook uses `float16` there and `bfloat16` on Ampere+ (A100, L4, ...).

## 0. Setup

In [1]:
!nvidia-smi

Sat Nov 14 08:07:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os, subprocess, sys
REPO_URL = "https://github.com/willychang21/mini-sglang"
BRANCH = "main"
if not os.path.exists("mini-sglang"):
    !git clone -b {BRANCH} {REPO_URL}
%cd mini-sglang
!git log --oneline -3
# Colab already ships a CUDA build of torch (with triton). Do not reinstall torch.
!pip install -q -e ".[test]" pyzmq
!python -c "import torch, triton; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'triton', triton.__version__)"

Cloning into 'mini-sglang'...
remote: Enumerating objects: 541, done.
remote: Counting objects: 100% (541/541), done.
remote: Compressing objects: 100% (353/353), done.
remote: Total 541 (delta 262), reused 423 (delta 155), pack-reused 0 (from 0)
Receiving objects: 100% (541/541), 5.43 MiB | 11.20 MiB/s, done.
Resolving deltas: 100% (262/262), done.
/content/mini-sglang
2314074 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #10 from willychang21/claude/notebook-main
af428b9 (origin/claude/notebook-main) Run the GPU notebook against main and add an Open in Colab link
bb17de8 Merge pull request #9 from willychang21/claude/rehearsal-verification
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for minisgl (pyproject.toml) ... done
torch 2.14.1+cu130 cuda 13.0 triton 3.8.0


In [3]:
from huggingface_hub import snapshot_download
MODEL = "/content/models/Qwen3-0.6B" if os.path.exists("/content") else os.path.abspath("models/Qwen3-0.6B")
snapshot_download("Qwen/Qwen3-0.6B", local_dir=MODEL)
os.environ["MINISGL_TEST_MODEL"] = MODEL  # enables tests/test_real_model.py and the real-model GPU test

import torch
major, minor = torch.cuda.get_device_capability()
DTYPE = "bfloat16" if major >= 8 else "float16"
print(torch.cuda.get_device_name(), f"sm_{major}{minor}", "->", DTYPE)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Tesla T4 sm_75 -> float16


## 1. Test suite on the GPU

`tests/test_gpu.py` only runs here. It checks

* CUDA Graph decode (triton backend, fp32, page_size 1 and 16) reproduces HuggingFace greedy output while the batch shrinks from 6 to 1 requests (padded to 8/4/2/1 with the dummy request),
* eager torch / triton backends on CUDA == HuggingFace,
* bf16 graph replay == bf16 eager,
* real Qwen3-0.6B, fp32, triton + CUDA Graph == HuggingFace.

`tests/test_triton_attention.py` now compiles the kernels for the GPU instead of running them in the Triton interpreter.

*What to expect*: everything passes; `test_tp.py` still uses Gloo on CPU.

In [4]:
!python -m pytest -q -x -rs

...........F
=================================== FAILURES ===================================
_______________________ test_padded_decode_matches_hf[1] _______________________

self = <triton.runtime.interpreter.GridExecutor object at 0x7d0bf4f99010>
args_dev = (tensor([[[-1.6232e+00, -9.3551e-01,  5.0552e-01,  ..., -1.0047e+00,
           1.8154e+00, -2.1834e-01],
         [-5...  ...,   0,   0,   0],
        [512, 512, 512,  ..., 512, 512, 512]]), tensor([0, 1, 2, 3, 4], dtype=torch.int32), ...)
kwargs = {'BLOCK_D': 16, 'BLOCK_M': 32, 'BLOCK_N': 32, 'DOT_FP32': False, ...}
argspec = FullArgSpec(args=['Q', 'K', 'V', 'O', 'PageTable', 'Rows', 'QoStart', 'ExtendLens', 'PrefixLens', 'sm_scale', 'stride_...texpr', 'BLOCK_D': 'tl.constexpr', 'BLOCK_N': 'tl.constexpr', 'PAGE_SIZE': 'tl.constexpr', 'DOT_FP32': 'tl.constexpr'})
args_hst = [tensor([[[-1.6232e+00, -9.3551e-01,  5.0552e-01,  ..., -1.0047e+00,
           1.8154e+00, -2.1834e-01],
         [-5...  ...,   0,   0,   0],
        [512,

## 2. Benchmarks

Every script takes `--device cuda --dtype ... --attention-backend {torch,triton} [--disable-cuda-graph]`.
The `torch` backend has no CUDA Graph support (its metadata shapes change every step), so it always runs eagerly.

In [5]:
CONFIGS = [  # (label, extra flags)
    ("torch, eager",          "--attention-backend torch --disable-cuda-graph"),
    ("triton, eager",         "--attention-backend triton --disable-cuda-graph"),
    ("triton, CUDA graph",    "--attention-backend triton"),
]
COMMON = f"--model {MODEL} --device cuda --dtype {DTYPE}"

def run_all(script, extra=""):
    for label, flags in CONFIGS:
        print(f"\n######## {script} | {label}\n", flush=True)
        !python benchmarks/{script} {COMMON} {flags} {extra}

### 2.1 Attention kernels + end-to-end decode (`bench_attention.py`)

Kernel tables: one attention call for one layer; torch = gather through the page table into a padded tensor + SDPA; triton = paged kernels, decode with `num_kv_splits` = 1/4/8/16.
End-to-end table: decode TPOT through the scheduler for torch / triton eager / triton + CUDA Graph.

*What to expect*
* decode: triton beats gather+SDPA, increasingly with kv_len (the gather copies all of K/V every layer); for bs=1 and long kv_len, splits > 1 is clearly faster than splits = 1 (one program per KV head cannot fill the GPU); for bs=64 splits matter little.
* prefill: torch SDPA (FlashAttention / mem-efficient kernels) may well win against this simple extend kernel; that is fine, prefill is compute-bound.
* end to end: CUDA Graph cuts decode TPOT a lot at small batch (Qwen3-0.6B decode is launch-bound: ~28 layers x ~15 kernels); the gap shrinks as bs grows.

In [6]:
!python benchmarks/bench_attention.py {COMMON}

device=cuda dtype=float16 attention_backend=default cuda_graph=on (CUDA only)

## Decode attention, one layer (ms, median)

| bs | kv_len | torch (gather+SDPA) | triton splits=1 | triton splits=4 | triton splits=8 | triton splits=16 |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 512 | 0.343 | 0.592 | 0.480 | 0.352 | 0.352 |
| 1 | 2048 | 0.610 | 1.614 | 0.807 | 0.784 | 0.818 |
| 1 | 8192 | 1.679 | 4.090 | 1.875 | 1.455 | 1.276 |
| 8 | 512 | 0.444 | 0.740 | 0.642 | 0.654 | 0.644 |
| 8 | 2048 | 1.188 | 1.926 | 1.693 | 1.602 | 1.565 |
| 8 | 8192 | 3.876 | 7.138 | 6.122 | 5.715 | 5.609 |
| 32 | 512 | 1.030 | 1.670 | 1.560 | 1.590 | 1.587 |
| 32 | 2048 | 3.720 | 6.202 | 5.671 | 5.619 | 5.572 |
| 32 | 8192 | 14.669 | 23.661 | 21.718 | 21.641 | 21.498 |
| 64 | 512 | 1.902 | 3.012 | 2.950 | 2.952 | 2.982 |
| 64 | 2048 | 7.301 | 11.317 | 10.993 | 10.886 | 10.913 |
| 64 | 8192 | 28.900 | 43.403 | 42.685 | 42.645 | 42.571 |

## Prefill attention (no prefix), one layer (ms, median)

| bs | q_len = kv

### 2.2 Ch4 KV cache (`bench_kv_cache.py`)
Uses `SequenceRunner` (prefill-phase batches), so CUDA Graph does not apply; the backend does.

*What to expect*: KV cache speedup grows with output length; `same output` is True.

In [7]:
run_all("bench_kv_cache.py")


######## bench_kv_cache.py | torch, eager

device=cuda dtype=float16 attention_backend=torch cuda_graph=off
[08:09:37 MainProcess INFO] Engine ready: Qwen3ForCausalLM layers=28 tp=0/1 device=cuda:0 dtype=torch.float16 attn=torch kv_tokens=4096 (448.1 MiB)
prompt_len=128
| output tokens | no cache: total s | no cache: TPOT ms | KV cache: total s | KV cache: TPOT ms | speedup | same output |
|---:|---:|---:|---:|---:|---:|:---:|
| 16 | 1.18 | 73.6 | 1.00 | 62.4 | 1.2x | True |
| 64 | 2.83 | 44.2 | 2.62 | 40.9 | 1.1x | False |
| 128 | 5.89 | 46.1 | 5.72 | 44.7 | 1.0x | False |

######## bench_kv_cache.py | triton, eager

device=cuda dtype=float16 attention_backend=triton cuda_graph=off
[08:10:05 MainProcess INFO] Engine ready: Qwen3ForCausalLM layers=28 tp=0/1 device=cuda:0 dtype=torch.float16 attn=triton kv_tokens=4096 (448.1 MiB)
prompt_len=128
| output tokens | no cache: total s | no cache: TPOT ms | KV cache: total s | KV cache: TPOT ms | speedup | same output |
|---:|---:|---:|---:|

### 2.3 Ch6/7 continuous batching (`bench_batching.py`)
*What to expect*: continuous batching gives several x the throughput of one-at-a-time; CUDA Graph helps the one-at-a-time row most (bs = 1 decode is purely launch-bound). `identical outputs` should be True in fp32; in bf16/fp16 an occasional difference between batch sizes is possible (different GEMM shapes, different rounding).

In [8]:
run_all("bench_batching.py")


######## bench_batching.py | torch, eager

device=cuda dtype=float16 attention_backend=torch cuda_graph=off
## Admission: conservative vs optimistic (retraction) under a tight KV budget

| workload | admission | KV tokens | wall s | output tok/s | peak batch | mean decode batch | retractions | identical |
|---|---|---:|---:|---:|---:|---:|---:|---|
[08:11:02 MainProcess INFO] Engine ready: Qwen3ForCausalLM layers=28 tp=0/1 device=cuda:0 dtype=torch.float16 attn=torch kv_tokens=1024 (112.1 MiB)
| exact lengths (ignore_eos) | conservative | 1024 | 7.2 | 171.4 | 12 | 9.2 | 0 | True |
[08:11:11 MainProcess INFO] Engine ready: Qwen3ForCausalLM layers=28 tp=0/1 device=cuda:0 dtype=torch.float16 attn=torch kv_tokens=1024 (112.1 MiB)
| exact lengths (ignore_eos) | optimistic | 1024 | 6.1 | 203.5 | 13 | 10.0 | 0 | True |
[08:11:19 MainProcess INFO] Engine ready: Qwen3ForCausalLM layers=28 tp=0/1 device=cuda:0 dtype=torch.float16 attn=torch kv_tokens=1024 (112.1 MiB)
| long exact outputs (ignor

### 2.4 Ch8 prefix cache (`bench_prefix_cache.py`)
*What to expect*: radix hit rate > 90%, much lower TTFT than naive; identical outputs.

In [9]:
run_all("bench_prefix_cache.py")


######## bench_prefix_cache.py | torch, eager

device=cuda dtype=float16 attention_backend=torch cuda_graph=off
12 requests, shared prefix 512 tokens

| cache | prompt tokens | computed prefill tokens | cache hit rate | wall s | mean TTFT s (sequential arrival) |
|---|---:|---:|---:|---:|---:|
[08:17:03 MainProcess INFO] Engine ready: Qwen3ForCausalLM layers=28 tp=0/1 device=cuda:0 dtype=torch.float16 attn=torch kv_tokens=16384 (1792.1 MiB)
| naive | 6376 | 6376 | 0.0% | 4.6 | 0.067 |
[08:17:10 MainProcess INFO] Engine ready: Qwen3ForCausalLM layers=28 tp=0/1 device=cuda:0 dtype=torch.float16 attn=torch kv_tokens=16384 (1792.1 MiB)
| radix | 6376 | 744 | 88.3% | 4.3 | 0.050 |

identical outputs: True

######## bench_prefix_cache.py | triton, eager

device=cuda dtype=float16 attention_backend=triton cuda_graph=off
12 requests, shared prefix 512 tokens

| cache | prompt tokens | computed prefill tokens | cache hit rate | wall s | mean TTFT s (sequential arrival) |
|---|---:|---:|---:|--

### 2.5 Ch10 speculative decoding (`bench_speculative.py`)
Uses `SequenceRunner`, so no CUDA Graph; compare torch vs triton only.

*What to expect*: "self (upper bound)" accepts 100%; layer-truncated drafts accept less; `lossless` True in fp32 (bf16 verification of k tokens at once vs one at a time can differ in rare near-ties).

In [10]:
run_all("bench_speculative.py", "--draft-layers 7,14")


######## bench_speculative.py | torch, eager

device=cuda dtype=float16 attention_backend=torch cuda_graph=off
[08:18:01 MainProcess INFO] Engine ready: Qwen3ForCausalLM layers=28 tp=0/1 device=cuda:0 dtype=torch.float16 attn=torch kv_tokens=2048 (224.1 MiB)
| task | draft | k | acceptance | tokens / target fwd | target fwds | draft fwds | tok/s | speedup | lossless |
|---|---|---:|---:|---:|---:|---:|---:|---:|:---:|
| open-ended | none (target only) | - | - | 1.00 | 64 | 0 | 21.2 | 1.00x | - |
[08:18:05 MainProcess INFO] Engine ready: Qwen3ForCausalLM layers=7 tp=0/1 device=cuda:0 dtype=torch.float16 attn=torch kv_tokens=2048 (56.0 MiB)
| open-ended | model: first 7 layers | 4 | 2% | 1.09 | 59 | 226 | 12.3 | 0.58x | True |
[08:18:12 MainProcess INFO] Engine ready: Qwen3ForCausalLM layers=14 tp=0/1 device=cuda:0 dtype=torch.float16 attn=torch kv_tokens=2048 (112.1 MiB)
| open-ended | model: first 14 layers | 4 | 5% | 1.21 | 53 | 202 | 8.6 | 0.41x | True |
| open-ended | n-gram prompt

### 2.6 Ch5 serving (`bench_serving.py`)
Starts the HTTP server once per configuration and drives it at several concurrencies.

*What to expect*: TPOT roughly flat with concurrency for triton + CUDA Graph until the batch gets large; higher with eager backends.

In [ ]:
import time, subprocess, httpx

SERVER_FLAGS = {
    "torch, eager":       ["--attention-backend", "torch", "--disable-cuda-graph"],
    "triton, eager":      ["--attention-backend", "triton", "--disable-cuda-graph"],
    "triton, CUDA graph": ["--attention-backend", "triton"],
}
for label, flags in SERVER_FLAGS.items():
    print(f"\n######## bench_serving.py | {label}\n", flush=True)
    server = subprocess.Popen([sys.executable, "-m", "minisgl", "--model", MODEL, "--device", "cuda",
                               "--dtype", DTYPE, "--port", "1919", *flags],
                              stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    try:
        for _ in range(300):  # wait until ready (graph capture takes a few seconds)
            try:
                if httpx.get("http://127.0.0.1:1919/v1/models", timeout=1).status_code == 200:
                    break
            except Exception:
                time.sleep(1)
        !python benchmarks/bench_serving.py --url http://127.0.0.1:1919 --concurrency 1,4,16 --num-requests 32
    finally:
        server.terminate(); server.wait()


######## bench_serving.py | torch, eager

| concurrency | requests | mean TTFT s | p99 TTFT s | mean TPOT ms | mean E2E s | output tok/s | req/s |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 1 | 32 | 0.06 | 0.32 | 44 | 1.41 | 22.6 | 0.71 |
| 4 | 32 | 0.08 | 0.16 | 50 | 1.63 | 78.7 | 2.46 |
| 16 | 32 | 0.16 | 0.22 | 68 | 2.28 | 224.4 | 7.01 |

######## bench_serving.py | triton, eager



## 3. `torch.profiler` trace of one decode step

We wrap `Engine.forward_batch` and profile exactly one decode step (batch of 8 requests, ~256 tokens of context) for each configuration, print the top CUDA kernels and save a Chrome trace (open in `chrome://tracing` or https://ui.perfetto.dev).

*What to expect*
* torch backend: per layer an `index` (gather) of the whole K/V history, mask construction, then an SDPA kernel;
* triton eager: `_decode_stage1_kernel` + `_decode_merge_kernel` per layer and no gather; many small launches with CPU gaps between them;
* triton + CUDA graph: the CPU side is a single `cudaGraphLaunch`, the GPU timeline is dense (no gaps).

In [ ]:
import torch
from torch.profiler import profile, ProfilerActivity
from minisgl import SamplingParams
from minisgl.llm import LLM

def profile_one_decode_step(backend, disable_graph, trace_path):
    llm = LLM(MODEL, load_tokenizer_=False, device="cuda", dtype=DTYPE, attention_backend=backend,
              disable_cuda_graph=disable_graph, max_total_tokens=16384, max_seq_len=2048, max_running_req=16)
    engine = llm.engine
    llm.generate([[1, 2, 3]] * 2, SamplingParams(max_tokens=4))  # warm-up / JIT
    orig = engine.forward_batch
    state = {"seen": 0, "prof": None}

    def wrapped(batch, args):
        if batch.is_decode and state["prof"] is None:
            state["seen"] += 1
            if state["seen"] == 3:  # skip the first decode steps
                torch.cuda.synchronize()
                with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
                    out = orig(batch, args)
                    torch.cuda.synchronize()
                state["prof"] = prof
                return out
        return orig(batch, args)

    engine.forward_batch = wrapped
    g = torch.Generator().manual_seed(0)
    prompts = torch.randint(100, 10000, (8, 256), generator=g).tolist()
    llm.generate(prompts, SamplingParams(max_tokens=8, ignore_eos=True))
    prof = state["prof"]
    prof.export_chrome_trace(trace_path)
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))
    llm.shutdown()
    return prof

for backend, disable_graph in [("torch", True), ("triton", True), ("triton", False)]:
    name = f"decode_{backend}_{'eager' if disable_graph else 'graph'}"
    print(f"\n######## {name}\n")
    profile_one_decode_step(backend, disable_graph, f"/content/{name}.json" if os.path.exists("/content") else f"{name}.json")

In [ ]:
# Download the traces (Colab)
try:
    from google.colab import files
    for n in ["decode_torch_eager", "decode_triton_eager", "decode_triton_graph"]:
        files.download(f"/content/{n}.json")
except ImportError:
    print("not on Colab: traces are in the working directory")

## 4. Notes to write back into the README

* Paste the tables from 2.1 (kernels + e2e) and the TPOT rows of 2.3 into `README.md` under a "GPU" section.
* If any `test_gpu.py` test fails, the most informative ones are `test_eager_fp32_matches_hf[triton]` (kernels) vs `test_cuda_graph_fp32_matches_hf` (graph padding / static buffers).